# 0.1 Prepare Notebook Environment

- Create a Jupyter Notebook: government_offices_data_transformation.ipynb inside /scripts.
- Ensure all required Python libraries are installed and imported: pandas, geopandas, shapely, osmnx, requests, json.

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
import requests
import json

# 0.2 Data Discovery and Fetch data

In [2]:
# Tags for different types of government offices
tags_list = [
    {"office": "government"},
    {"office": "administrative"},
    {"amenity": "townhall"},
    {"amenity": "public_building"},
    {"office": "employment_agency"},
]

# A list to store the data we fetch
all_offices = []

# Step 1: Fetch data for each tag
for tags in tags_list:
    try:
        gdf = ox.features_from_place("Berlin, Germany", tags)
        all_offices.append(gdf)
    except Exception:
        pass  # If no data found for a tag, just skip it

# Step 2: Combine all data into one dataframe
if all_offices:
    government_offices_gdf = pd.concat(all_offices, ignore_index=False)

    # Reset index so 'osmid' (if it exists) becomes a column
    government_offices_gdf = government_offices_gdf.reset_index()

    # Step 3: Remove duplicate entries
    if 'osmid' in government_offices_gdf.columns:
        government_offices_gdf = government_offices_gdf.drop_duplicates(subset=['osmid'])
    elif {'element_type', 'osmid'}.issubset(government_offices_gdf.columns):
        government_offices_gdf = government_offices_gdf.drop_duplicates(subset=['element_type', 'osmid'])
    else:
        government_offices_gdf = government_offices_gdf.drop_duplicates()

    # Step 4: Optionally filter offices with German government-related names
    german_keywords = [
        'Bürgeramt', 'Bezirksamt', 'Finanzamt', 'Standesamt',
        'Sozialamt', 'Jobcenter', 'Ausländerbehörde',
        'Landesamt', 'Senatsverwaltung', 'Ordnungsamt'
    ]

    if 'name' in government_offices_gdf.columns:
        german_offices = government_offices_gdf[
            government_offices_gdf['name'].str.contains('|'.join(german_keywords), case=False, na=False)
        ]
    else:
        german_offices = pd.DataFrame()  # Empty if no 'name' column found

    # Step 5: Show how many results were found
    print(f"Total unique government office entries: {len(government_offices_gdf)}")
    print(f"Total unique German government office entries: {len(german_offices)}")

else:
    print("No government office data found.")


Total unique government office entries: 441
Total unique German government office entries: 123


# 0.3 Basic data overview

In [3]:
# Display basic dataset information

if government_offices_gdf is not None:
    print(f"Dataset Shape: {government_offices_gdf.shape}")
    print(f"Total Columns: {len(government_offices_gdf.columns)}")
    display(government_offices_gdf.head())


Dataset Shape: (441, 177)
Total Columns: 177


,element,id,geometry,addr:city,addr:housenumber,addr:postcode,addr:street,government,level,name,...,historic_name:de,historic_name:en,building:parts,surveillance,name:prefix,source:website,ele,brand,brand:wikipedia,internet_access:ssid
0,node,331398399,POINT (13.34756 52.42794),Berlin,87,12249,Gallwitzallee,public_service,1,Bürgeramt Lankwitz,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,356925920,POINT (13.32987 52.50927),Berlin,87,10623,Fasanenstraße,NaN,NaN,Bundesanstalt für Immobilienaufgaben,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,361001240,POINT (13.26794 52.50944),Berlin,12-14,14052,Heerstraße,public_service,NaN,Bürgeramt Heerstraße,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,371520086,POINT (13.18393 52.51935),Berlin,25-30,13593,Wilhelmstraße,NaN,NaN,Bundesanstalt für Geowissenschaften und Rohsto...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,434480600,POINT (13.20134 52.53527),Berlin,1,13597,Am Wall,register_office,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 0.4: Column Completeness Analysis

The dataset contains **441 records** and **177 columns** describing government offices in Berlin.  
Key structural fields (`element`, `id`, `geometry`) are **fully complete (0% nulls)**, ensuring data integrity for spatial analysis.  

#### 📊 Overall Data Quality
- **Total Columns:** 177  
- **Columns with >85% missing values:** 155 (≈88%)  
- **Columns with <15% missing values (key usable fields):** 19  
- **German-named offices identified:** 123  

#### ✅ Core Columns (High Completeness)
| Column | Completeness | Description |
|--------|---------------|-------------|
| `name` | 95.9% | Office name |
| `office` | 95.7% | Office type (e.g., government, administrative) |
| `addr:street` | 73.7% | Street name |
| `addr:housenumber` | 74.4% | House number |
| `addr:postcode` | 72.3% | Postal code |
| `addr:city` | 71.9% | City name (mostly Berlin) |
| `government` | 52.8% | Government classification |
| `addr:suburb` | 51.2% | District or neighborhood |
| `wheelchair` | 44.0% | Accessibility information |
| `website` | 38.5% | Official website URL |
| `opening_hours` | 30.4% | Public opening times |

#### ⚠️ Sparse Columns (Low Completeness)
Most auxiliary fields (e.g., `email`, `contact:phone`, `heritage`, `architect`, multilingual names, and `social media` tags) show **>85–99% null values**, indicating limited or optional tagging in OpenStreetMap entries.

#### 🧠 Data Schema Summary
The refined schema retains **19 key fields**, balancing location, identity, and accessibility attributes.  
These include essential identifiers (`office_id`, `id`, `geometry`), contact and address details, and accessibility metadata (`wheelchair`, `opening_hours`).

#### 🧩 Key Insights
- Spatial and identifying data are robust and complete.
- Contact, heritage, and descriptive metadata are often missing.
- The dataset is suitable for **location-based mapping and government office classification**, but requires external enrichment for detailed service attributes.

In [4]:
# Analyze how many values are missing in each column

if government_offices_gdf is not None:
    missing_analysis = pd.DataFrame({
        'Column': government_offices_gdf.columns,
        'Non-Null Count': government_offices_gdf.count(),
        'Null Count': government_offices_gdf.isnull().sum(),
        'Null Percentage': (government_offices_gdf.isnull().sum() / len(government_offices_gdf) * 100).round(2)
    }).sort_values('Null Percentage')

    display(missing_analysis)

    high_missing = missing_analysis[missing_analysis['Null Percentage'] > 85]
    print(f"Columns with >85% missing values: {len(high_missing)}")
    if len(high_missing) > 0:
        for col in high_missing['Column']:
            print(f" - {col}")

,Column,Non-Null Count,Null Count,Null Percentage
element,element,441,0,0.00
id,id,441,0,0.00
geometry,geometry,441,0,0.00
name,name,423,18,4.08
office,office,422,19,4.31
...,...,...,...,...
reservation,reservation,1,440,99.77
contact:linkedin,contact:linkedin,1,440,99.77
contact:tiktok,contact:tiktok,1,440,99.77
old_name:de,old_name:de,1,440,99.77


Columns with >85% missing values: 155
 - short_name
 - check_date
 - contact:phone
 - check_date:opening_hours
 - toilets:wheelchair
 - roof:shape
 - type
 - email
 - start_date
 - operator:wikidata
 - opening_hours:signed
 - name:en
 - brand:wikidata
 - amenity
 - old_name
 - contact:email
 - roof:levels
 - brand
 - operator:short
 - source
 - heritage
 - heritage:operator
 - ref:lda
 - lda:criteria
 - landuse
 - description
 - branch
 - level
 - name:de
 - name:pl
 - townhall:type
 - wheelchair:description
 - contact:fax
 - ref:bufa
 - contact:mastodon
 - fax
 - addr:housename
 - note
 - alt_name
 - operator:wikipedia
 - image
 - building:colour
 - official_name
 - roof:colour
 - name:es
 - name:ru
 - name:fr
 - name:ja
 - addr:floor
 - social_security
 - wikimedia_commons
 - brand:wikipedia
 - operator:type
 - architect
 - building:material
 - heritage:website
 - tourism
 - internet_access
 - internet_access:fee
 - smoking
 - thw:lv
 - research_institution
 - addr:place
 - barrier
 

In [5]:
# Explore all columns 
government_offices_gdf.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
element,441,3,node,271,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id,441.0,NaN,NaN,NaN,4137908020.07483,4527969531.093439,3343.0,104110908.0,2066425453.0,7609552662.0,13247532181.0
geometry,441,440,POINT (13.3692258 52.5282678),2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
addr:city,317,1,Berlin,317,NaN,NaN,NaN,NaN,NaN,NaN,NaN
addr:housenumber,328,169,1,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
source:website,1,1,https://scharfstein-group.com/frankfurter-alle...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ele,1,1,100,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand,27,2,Jobcenter,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand:wikipedia,7,1,de:Bundesagentur für Arbeit,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
#expland all columns to see more details
pd.set_option('display.max_columns', None)
#print(government_offices_gdf.head(3))

# 0.5 Identify key columns

In [14]:
if government_offices_gdf is not None:
    desired_columns = [
        'osmid', 'element_type', 'name', 'office', 'amenity',
        'addr:street', 'addr:housenumber', 'addr:postcode',
        'addr:city', 'addr:suburb', 'addr:district',
        'phone', 'contact:phone', 'email', 'contact:email',
        'website', 'contact:website', 'opening_hours',
        'wheelchair', 'government', 'operator', 'geometry'
    ]

    available_key_columns = [col for col in desired_columns if col in government_offices_gdf.columns]

    print(f"Available key columns ({len(available_key_columns)}):")
    print(f"non-null counts and percentages:")
    for col in available_key_columns:
        non_null = government_offices_gdf[col].count()
        percentage = (non_null / len(government_offices_gdf) * 100).round(1)
        print(f" - {col}: {non_null} ({percentage}%)")

Available key columns (19):
non-null counts and percentages:
 - name: 423 (95.9%)
 - office: 422 (95.7%)
 - amenity: 29 (6.6%)
 - addr:street: 325 (73.7%)
 - addr:housenumber: 328 (74.4%)
 - addr:postcode: 319 (72.3%)
 - addr:city: 317 (71.9%)
 - addr:suburb: 226 (51.2%)
 - phone: 91 (20.6%)
 - contact:phone: 55 (12.5%)
 - email: 42 (9.5%)
 - contact:email: 28 (6.3%)
 - website: 170 (38.5%)
 - contact:website: 98 (22.2%)
 - opening_hours: 134 (30.4%)
 - wheelchair: 194 (44.0%)
 - government: 233 (52.8%)
 - operator: 86 (19.5%)
 - geometry: 441 (100.0%)


# 0.6 Finding German Government Office Names

In [15]:
# Filter offices with German government-related names

if government_offices_gdf is not None and 'name' in government_offices_gdf.columns:
    german_keywords = [
        'Bürgeramt', 'Bezirksamt', 'Finanzamt', 'Standesamt',
        'Sozialamt', 'Jobcenter', 'Ausländerbehörde',
        'Landesamt', 'Senatsverwaltung', 'Ordnungsamt'
    ]

    german_offices = government_offices_gdf[
        government_offices_gdf['name'].str.contains('|'.join(german_keywords), case=False, na=False)
    ]

    print(f"German-named offices found: {len(german_offices)}")
    print(german_offices['name'].value_counts().head(10))


German-named offices found: 123
name
Standesamt                                                         5
Jobcenter                                                          4
Bürgeramt                                                          3
Senatsverwaltung für Mobilität, Verkehr, Klimaschutz und Umwelt    2
Bürgeramt Köpenick                                                 2
Senatsverwaltung für Bildung, Jugend und Familie                   2
Bürgeramt 2                                                        2
Finanzamt Mitte/Tiergarten                                         2
Landesamt für Mess- und Eichwesen Berlin-Brandenburg               1
Bezirksamt - Facility Management                                   1
Name: count, dtype: int64


### Data types analyis

In [16]:
# Check data types of all columns

if government_offices_gdf is not None:
    print(government_offices_gdf.dtypes.value_counts())

object      175
int64         1
geometry      1
Name: count, dtype: int64


# 0.7 Planned Table Schema

In [17]:
schema = {
    'office_id': 'INT (Primary Key) - Unique OSM ID',
    'district_id': 'INT (Foreign Key) - Link to districts table',
    'name': 'TEXT - Office name',
    'office_type': 'TEXT - Type of government office',
    'street': 'TEXT - Street name',
    'housenumber': 'TEXT - House number',
    'postal_code': 'TEXT - Postal code',
    'district': 'TEXT - District name',
    'neighborhood': 'TEXT - Neighborhood name',
    'phone_number': 'TEXT - Contact phone',
    'email': 'TEXT - Contact email',
    'website': 'TEXT - Website URL',
    'services_offered': 'TEXT - Services provided',
    'appointment_required': 'TEXT - Appointment necessity',
    'openinghours': 'TEXT - Opening hours',
    'wheelchair_accessible': 'TEXT - Accessibility info',
    'latitude': 'FLOAT - Latitude coordinate',
    'longitude': 'FLOAT - Longitude coordinate',
    'coordinate': 'TEXT - Geometry type'
}

for i, (col, desc) in enumerate(schema.items(), 1):
    print(f"{i:2d}. {col:25s} - {desc}")

 1. office_id                 - INT (Primary Key) - Unique OSM ID
 2. district_id               - INT (Foreign Key) - Link to districts table
 3. name                      - TEXT - Office name
 4. office_type               - TEXT - Type of government office
 5. street                    - TEXT - Street name
 6. housenumber               - TEXT - House number
 7. postal_code               - TEXT - Postal code
 8. district                  - TEXT - District name
 9. neighborhood              - TEXT - Neighborhood name
10. phone_number              - TEXT - Contact phone
11. email                     - TEXT - Contact email
12. website                   - TEXT - Website URL
13. services_offered          - TEXT - Services provided
14. appointment_required      - TEXT - Appointment necessity
15. openinghours              - TEXT - Opening hours
16. wheelchair_accessible     - TEXT - Accessibility info
17. latitude                  - FLOAT - Latitude coordinate
18. longitude                 - 

# 0.8: Prepare /sources Directory

In [18]:
# Prepare and save sample data for documentation

if government_offices_gdf is not None:
    sample_size = min(100, len(government_offices_gdf))
    sample_df = government_offices_gdf.head(sample_size)
    print(f"Sample data prepared ({sample_size} rows)")

Sample data prepared (100 rows)


# 0.9 Review and Checklist

In [19]:
# Final review of Step 1 process

if government_offices_gdf is not None:
    print("✅ STEP 0 COMPLETION CHECKLIST:")
    print(f" - Data fetched successfully ({len(government_offices_gdf)} unique entries)")
    print(" - Data structure analyzed")
    print(" - Key columns identified")
    print(" - Schema designed (18 columns)")
    print(" - Sample data prepared")
else:
    print("⚠️ STEP 0 INCOMPLETE - No data retrieved")


✅ STEP 0 COMPLETION CHECKLIST:
 - Data fetched successfully (441 unique entries)
 - Data structure analyzed
 - Key columns identified
 - Schema designed (18 columns)
 - Sample data prepared


# 1.1 Data Cleaning

Standardize Column Names and Formats:

- Normalize all columns to snake_case and lowercase.
- Map source-specific field names to a unified schema (e.g., office_name, department, address, lat, lon, phone, website).

Validate and Clean Geospatial Data:

- Ensure all geometries are valid and in a consistent CRS (EPSG:4326 / WGS84).
- Remove duplicate entries based on name and centroid proximity (~5–10m).
- Explode multipart geometries where necessary.
